# Discussion 3: Pandas + EDA exam prep

This notebook contains the `income` and `properties` `DataFrame`s from the discussion 3 worksheet. These `DataFrame`s are completely made-up for teaching demonstration purposes.

It’s the annual Monopoly World Championship! The finalists: Shawn, Amanda, Neil, and Annie are playing Monopoly, a board game where players pay a price to buy properties, which can then generate income for them. Each property can be owned by only one player at a time. At the end of the game, the player with the most money wins. 

Shawn wants to figure out which properties are most worth buying. He creates a `DataFrame` called `income` with data on the current game state, shown on the left. He also finds a `DataFrame` called `properties` with data on Monopoly properties, shown on the right. Both tables have 28 rows. For brevity, only the first few rows of each `DataFrame` are shown.

<div align="center">
    <img src = "exam_prep1.png" width = "600">

In [1]:
# Run this to read the DataFrames
import pandas as pd

income = pd.read_csv("income.csv")
properties = pd.read_csv("properties.csv")
display(income.head())
display(properties.head())

,Player,Property,Income Generated
0,Shawn,Boardwalk,$425
1,Amanda,Park Place,$375
2,Neil,Marvin Gardens,$200
3,NaN,Kentucky Ave,NaN
4,Shawn,Pennsylvania Ave,$150


,Property,Property Color,Purchase Price
0,Mediterranean Ave,Brown,60
1,Baltic Ave,Brown,60
2,Oriental Ave,Light Blue,100
3,Vermont Ave,Light Blue,100
4,Connecticut Ave,Light Blue,120


## Part C

In [2]:
# Answer: the last option, income.groupby("Player").size()
#
#   income.groupby("Player").count()          -> a DataFrame, one count column per
#       remaining column, not the single Series asked for.
#   income.groupby("Player").value_counts()   -> counts of distinct *rows* within
#       each player, giving a MultiIndexed Series.
#   income["Player", "Property"]              -> TypeError. Selecting several columns
#       needs a list: income[["Player", "Property"]].
#   income.groupby("Player").size()           -> one row per player, value = number of
#       properties they own. Note groupby drops the NaN "Player" rows (the unowned
#       properties) by default, which is what we want here.

income.groupby("Player").size()

Player
Amanda    7
Annie     6
Neil      4
Shawn     6
dtype: int64

## Part D

In [3]:
# "Income Generated" is stored as a *string* ("$425"), so it must be stripped of the
# dollar sign and cast to a number before any arithmetic will work.
income["Income Generated"] = income["Income Generated"].str.strip("$").astype(float)

# Total income each player has earned so far, largest first.
income.groupby("Player")["Income Generated"].sum().sort_values(ascending=False)

Player
Amanda    2585.0
Annie     2025.0
Shawn     1725.0
Neil      1450.0
Name: Income Generated, dtype: float64

## Part E

In [4]:
# Both tables key on "Property", so that is the join column. The default inner join
# keeps only properties present in *both* tables: the income table also lists board
# squares like "Jail" and "Free Parking" that cannot be bought, and the properties
# table lists ones nobody has landed on yet, so 24 of the 28 rows survive.
combined_df = income.merge(properties, on="Property")
combined_df

,Player,Property,Income Generated,Property Color,Purchase Price
0,Shawn,Boardwalk,425.0,Dark Blue,400
1,Amanda,Park Place,375.0,Dark Blue,350
2,Neil,Marvin Gardens,200.0,Yellow,280
3,NaN,Kentucky Ave,NaN,Red,220
4,Shawn,Pennsylvania Ave,150.0,Green,320
5,Annie,Oriental Ave,50.0,Light Blue,100
6,Amanda,Baltic Ave,60.0,Brown,60
7,Shawn,Mediterranean Ave,200.0,Brown,60
8,NaN,Reading Railroad,NaN,Railroad,200
9,Amanda,Vermont Ave,200.0,Light Blue,100


In [5]:
# Profit = what the property has earned so far, minus what it cost to buy.
# Rows for unowned properties carry NaN income, so their profit is NaN too.
combined_df["Profit"] = combined_df["Income Generated"] - combined_df["Purchase Price"]
combined_df

,Player,Property,Income Generated,Property Color,Purchase Price,Profit
0,Shawn,Boardwalk,425.0,Dark Blue,400,25.0
1,Amanda,Park Place,375.0,Dark Blue,350,25.0
2,Neil,Marvin Gardens,200.0,Yellow,280,-80.0
3,NaN,Kentucky Ave,NaN,Red,220,NaN
4,Shawn,Pennsylvania Ave,150.0,Green,320,-170.0
5,Annie,Oriental Ave,50.0,Light Blue,100,-50.0
6,Amanda,Baltic Ave,60.0,Brown,60,0.0
7,Shawn,Mediterranean Ave,200.0,Brown,60,140.0
8,NaN,Reading Railroad,NaN,Railroad,200,NaN
9,Amanda,Vermont Ave,200.0,Light Blue,100,100.0


## Part F

In [6]:
# A function passed to .apply() on a groupby receives each group as a sub-DataFrame
# and returns one value (or row) per group. Here we summarise each color group by its
# average profit.
def func(group):
    return group["Profit"].mean()

In [7]:
# Group the merged table by color group and apply the function above, then rank the
# color groups from most to least profitable.
combined_df.groupby("Property Color").apply(func).sort_values(ascending=False)

/var/folders/86/33q466997r1flrlvc587dg9h0000gn/T/ipykernel_88768/1055985987.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_df.groupby("Property Color").apply(func).sort_values(ascending=False)


Property Color
Orange        271.666667
Pink          170.000000
Light Blue    143.333333
Red           110.000000
Yellow         80.000000
Railroad       75.000000
Brown          70.000000
Utility        50.000000
Dark Blue      25.000000
Green         -60.000000
dtype: float64